# Longformer: Long Document Transformer

Longformer, introduced by Beltagy et al. (2020), addresses the challenge of scaling Transformers to long documents by introducing an efficient attention mechanism. Here, we'll delve into the mathematical details of this mechanism.

## 1. Standard Self-Attention

In the standard self-attention mechanism used by models like BERT, the attention score between two tokens is computed as follows:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

where:
- $Q \in \mathbb{R}^{n \times d_k}$ is the query matrix,
- $K \in \mathbb{R}^{n \times d_k}$ is the key matrix,
- $V \in \mathbb{R}^{n \times d_v}$ is the value matrix,
- $n$ is the sequence length,
- $d_k$ and $d_v$ are the dimensions of the keys/queries and values, respectively.

The computational complexity of this operation is $O(n^2)$ due to the $QK^T$ matrix multiplication, which becomes prohibitive for long sequences.

## 2. Longformer's Attention Mechanism

Longformer introduces two main modifications: sliding window attention and global attention.

### Sliding Window Attention

Sliding window attention restricts each token to attend to a fixed-size local window around it. If the window size is $w$, each token attends to $w$ tokens instead of $n$.

Mathematically, for each token $i$:

$$
\text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left(\frac{Q_i K_{i-w/2:i+w/2}^T}{\sqrt{d_k}}\right) V_{i-w/2:i+w/2}
$$

Here, $Q_i$, $K_{i-w/2:i+w/2}$, and $V_{i-w/2:i+w/2}$ are the query, key, and value vectors within the local window centered at $i$.

The computational complexity is reduced to $O(nw)$.

### Global Attention

Global attention allows a small subset of tokens (e.g., special tokens like [CLS]) to attend to all tokens in the sequence, facilitating information flow across the entire sequence.

Let $G$ be the set of global tokens. The attention for a global token $g \in G$ is computed as:

$$
\text{Attention}(Q_g, K, V) = \text{softmax}\left(\frac{Q_g K^T}{\sqrt{d_k}}\right) V
$$

The global tokens have a computational complexity of $O(|G|n)$.

## 3. Combined Attention

Combining both types of attention, the overall attention mechanism for each token is:

$$
\text{CombinedAttention}(Q, K, V) = \text{WindowedAttention}(Q, K, V) + \text{GlobalAttention}(Q, K, V)
$$

For tokens not in $G$:

$$
\text{Attention}_i = \text{softmax}\left(\frac{Q_i K_{i-w/2:i+w/2}^T}{\sqrt{d_k}}\right) V_{i-w/2:i+w/2}
$$

For tokens in $G$:

$$
\text{Attention}_g = \text{softmax}\left(\frac{Q_g K^T}{\sqrt{d_k}}\right) V
$$

## 4. Complexity Analysis

The combined complexity for a sequence with $n$ tokens, window size $w$, and $|G|$ global tokens is:

$$
O(nw + |G|n)
$$

For typical values $w \ll n$ and $|G| \ll n$, the complexity is much lower than the $O(n^2)$ complexity of the standard self-attention.

## References

1. Beltagy, I., Peters, M. E., & Cohan, A. (2020). Longformer: The Long-Document Transformer. arXiv preprint arXiv:2004.05150. Available at: [https://arxiv.org/abs/2004.05150](https://arxiv.org/abs/2004.05150)
2. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). Attention is All You Need. Advances in Neural Information Processing Systems, 30, 5998-6008. Available at: [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

In [ ]:
# Cell 1: Setup and Dependencies
print("Setting up Longformer implementation...")

# Install packages with better compatibility
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

try:
    install_package("torch")
    install_package("matplotlib")
    install_package("seaborn")
    install_package("numpy")
    print("✓ Packages installed successfully")
except Exception as e:
    print(f"Installation warning: {e}")

# Import packages
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from typing import Optional, Tuple, List
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
np.random.seed(42)

In [ ]:
# Cell 2: Efficient Longformer Implementation
print("Creating efficient Longformer implementation...")

class EfficientLongformerAttention(nn.Module):
    """
    Efficient implementation of Longformer attention with:
    - Sliding window attention
    - Global attention for special tokens
    - Fixed tensor dimension handling
    """
    def __init__(self, d_model, n_heads, window_size, global_tokens=None, dilation=1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.dilation = dilation
        self.global_tokens = global_tokens or []
        
        # Linear projections
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        
        # Scale factor
        self.scale = 1.0 / math.sqrt(self.head_dim)
        
    def _sliding_window_attention(self, q, k, v, attention_mask=None):
        """Efficient sliding window attention with fixed dimensions"""
        batch_size, n_heads, seq_len, head_dim = q.shape
        
        # Create output tensor
        output = torch.zeros_like(q)
        
        # Process each position
        for i in range(seq_len):
            # Calculate window boundaries
            start_idx = max(0, i - self.window_size // 2)
            end_idx = min(seq_len, i + self.window_size // 2 + 1)
            
            # Extract local keys and values
            local_k = k[:, :, start_idx:end_idx, :]  # [batch, heads, window_len, head_dim]
            local_v = v[:, :, start_idx:end_idx, :]  # [batch, heads, window_len, head_dim]
            
            # Query for current position
            curr_q = q[:, :, i:i+1, :]  # [batch, heads, 1, head_dim]
            
            # Compute attention scores
            scores = torch.matmul(curr_q, local_k.transpose(-2, -1)) * self.scale
            # scores: [batch, heads, 1, window_len]
            
            # Apply attention mask if provided
            if attention_mask is not None:
                local_mask = attention_mask[:, start_idx:end_idx].unsqueeze(1).unsqueeze(2)
                scores = scores.masked_fill(local_mask == 0, float('-inf'))
            
            # Softmax
            attn_weights = F.softmax(scores, dim=-1)
            
            # Apply attention to values
            attended = torch.matmul(attn_weights, local_v)  # [batch, heads, 1, head_dim]
            output[:, :, i, :] = attended.squeeze(2)
        
        return output, None  # Return None for weights to save memory
    
    def _global_attention(self, q, k, v, global_indices, attention_mask=None):
        """Compute global attention for specified token positions"""
        if not global_indices:
            return torch.zeros_like(q), None
        
        batch_size, n_heads, seq_len, head_dim = q.shape
        output = torch.zeros_like(q)
        
        # Process global tokens
        for global_idx in global_indices:
            if global_idx >= seq_len:
                continue
                
            # Global query
            global_q = q[:, :, global_idx:global_idx+1, :]  # [batch, heads, 1, head_dim]
            
            # Compute attention scores with all positions
            scores = torch.matmul(global_q, k.transpose(-2, -1)) * self.scale
            # scores: [batch, heads, 1, seq_len]
            
            # Apply mask if provided
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(1).unsqueeze(2)  # [batch, 1, 1, seq_len]
                scores = scores.masked_fill(mask == 0, float('-inf'))
            
            # Softmax
            attn_weights = F.softmax(scores, dim=-1)
            
            # Apply to values
            global_output = torch.matmul(attn_weights, v)  # [batch, heads, 1, head_dim]
            output[:, :, global_idx, :] = global_output.squeeze(2)
        
        return output, None
    
    def forward(self, x, attention_mask=None, global_attention_mask=None):
        """
        Forward pass of Longformer attention
        
        Args:
            x: Input tensor [batch_size, seq_len, d_model]
            attention_mask: Mask for padding tokens [batch_size, seq_len]
            global_attention_mask: Mask for global tokens [batch_size, seq_len]
        """
        batch_size, seq_len, _ = x.shape
        
        # Linear projections
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Sliding window attention
        sliding_output, sliding_weights = self._sliding_window_attention(q, k, v, attention_mask)
        
        # Global attention
        global_indices = []
        if global_attention_mask is not None:
            # Get global indices for the first item in batch (assuming same pattern)
            global_indices = torch.nonzero(global_attention_mask[0], as_tuple=False).squeeze(-1).tolist()
            if isinstance(global_indices, int):
                global_indices = [global_indices]
        
        global_output, global_weights = self._global_attention(q, k, v, global_indices, attention_mask)
        
        # Combine sliding window and global attention
        # For global tokens, use global attention; for others, use sliding window
        combined_output = sliding_output.clone()
        if global_indices:
            for idx in global_indices:
                if idx < seq_len:
                    combined_output[:, :, idx, :] = global_output[:, :, idx, :]
        
        # Reshape and project
        combined_output = combined_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model
        )
        
        output = self.out_proj(combined_output)
        
        return output, sliding_weights, global_weights

class LongformerLayer(nn.Module):
    """Complete Longformer transformer layer"""
    def __init__(self, d_model, n_heads, d_ff, window_size, dropout=0.1, dilation=1):
        super().__init__()
        
        self.attention = EfficientLongformerAttention(d_model, n_heads, window_size, dilation=dilation)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x, attention_mask=None, global_attention_mask=None):
        # Self-attention with residual connection
        attn_output, sliding_weights, global_weights = self.attention(
            x, attention_mask, global_attention_mask
        )
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)
        
        return x, sliding_weights, global_weights

class EfficientLongformer(nn.Module):
    """Complete Longformer model for sequence classification"""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, window_size, 
                 max_seq_len=4096, num_classes=2, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.window_size = window_size
        self.max_seq_len = max_seq_len
        
        # Token and position embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        
        # Transformer layers
        self.layers = nn.ModuleList([
            LongformerLayer(d_model, n_heads, d_ff, window_size, dropout, dilation=1)
            for i in range(n_layers)
        ])
        
        # Classification head
        self.pooler = nn.Linear(d_model, d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        
    def forward(self, input_ids, attention_mask=None, global_attention_mask=None):
        batch_size, seq_len = input_ids.shape
        
        # Create position IDs
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(position_ids)
        x = self.embedding_dropout(token_emb + pos_emb)
        
        # Store attention weights for visualization
        all_sliding_weights = []
        all_global_weights = []
        
        # Pass through transformer layers
        for layer in self.layers:
            x, sliding_weights, global_weights = layer(x, attention_mask, global_attention_mask)
            all_sliding_weights.append(sliding_weights)
            all_global_weights.append(global_weights)
        
        # Classification
        # Use [CLS] token (first token) for classification
        cls_output = x[:, 0]  # [batch_size, d_model]
        pooled_output = torch.tanh(self.pooler(cls_output))
        logits = self.classifier(pooled_output)
        
        return {
            'logits': logits,
            'last_hidden_state': x,
            'sliding_attention_weights': all_sliding_weights,
            'global_attention_weights': all_global_weights
        }

print("✓ Fixed Longformer implementation created successfully")

In [ ]:
# Cell 3: Dataset and Utilities
print("Creating dataset and utility functions...")

class LongSequenceDataset(Dataset):
    """Dataset for long sequence classification tasks"""
    def __init__(self, num_samples, seq_len, vocab_size, num_classes=2):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.num_classes = num_classes
        
        # Generate synthetic long sequences with patterns
        self.data = []
        self.labels = []
        
        for _ in range(num_samples):
            # Create sequence with local and global patterns
            sequence = self._generate_sequence()
            label = self._generate_label(sequence)
            
            self.data.append(sequence)
            self.labels.append(label)
    
    def _generate_sequence(self):
        """Generate a sequence with meaningful patterns"""
        sequence = torch.zeros(self.seq_len, dtype=torch.long)
        
        # Add [CLS] token at the beginning
        sequence[0] = 1  # CLS token
        
        # Add local patterns (short-range dependencies)
        for i in range(1, self.seq_len - 1, 10):
            pattern_len = min(5, self.seq_len - i)
            pattern = torch.randint(2, self.vocab_size // 2, (pattern_len,))
            sequence[i:i+pattern_len] = pattern
        
        # Add global patterns (long-range dependencies)
        # Insert special tokens at specific positions
        global_positions = [self.seq_len // 4, self.seq_len // 2, 3 * self.seq_len // 4]
        global_token = self.vocab_size - 1  # Special global token
        
        for pos in global_positions:
            if pos < self.seq_len:
                sequence[pos] = global_token
        
        # Fill remaining positions with random tokens
        mask = sequence == 0
        sequence[mask] = torch.randint(2, self.vocab_size - 1, (mask.sum(),))
        
        return sequence
    
    def _generate_label(self, sequence):
        """Generate label based on sequence patterns"""
        # Label based on presence of global tokens and local patterns
        global_token = self.vocab_size - 1
        global_count = (sequence == global_token).sum().item()
        
        # Simple rule: positive if many global tokens and specific patterns
        local_pattern_score = (sequence[1:6] > self.vocab_size // 2).sum().item()
        
        if global_count >= 2 and local_pattern_score >= 3:
            return 1
        else:
            return 0
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

def create_attention_masks(input_ids, global_token_id=None):
    """Create attention masks for Longformer"""
    batch_size, seq_len = input_ids.shape
    
    # Regular attention mask (no padding in this synthetic data)
    attention_mask = torch.ones_like(input_ids)
    
    # Global attention mask
    global_attention_mask = torch.zeros_like(input_ids)
    
    # Set global attention for [CLS] token and special global tokens
    global_attention_mask[:, 0] = 1  # CLS token
    
    if global_token_id is not None:
        global_positions = (input_ids == global_token_id)
        global_attention_mask[global_positions] = 1
    
    return attention_mask, global_attention_mask

def visualize_attention_patterns():
    """Visualize different attention patterns"""
    print("Attention Pattern Comparison")
    print("=" * 50)
    
    seq_len = 64
    window_size = 16
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Standard self-attention
    std_attention = torch.ones(seq_len, seq_len)
    std_attention = torch.tril(std_attention)  # Causal mask for autoregressive
    
    im1 = axes[0, 0].imshow(std_attention.numpy(), cmap='Blues', aspect='auto')
    axes[0, 0].set_title('Standard Self-Attention\n(Full O(n²) complexity)')
    axes[0, 0].set_xlabel('Key Positions')
    axes[0, 0].set_ylabel('Query Positions')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Sliding window attention
    sliding_attention = torch.zeros(seq_len, seq_len)
    for i in range(seq_len):
        start = max(0, i - window_size // 2)
        end = min(seq_len, i + window_size // 2 + 1)
        sliding_attention[i, start:end] = 1
    
    im2 = axes[0, 1].imshow(sliding_attention.numpy(), cmap='Blues', aspect='auto')
    axes[0, 1].set_title(f'Sliding Window Attention\n(Window size: {window_size})')
    axes[0, 1].set_xlabel('Key Positions')
    axes[0, 1].set_ylabel('Query Positions')
    plt.colorbar(im2, ax=axes[0, 1])
    
    # Longformer attention (sliding + global)
    longformer_attention = sliding_attention.clone()
    
    # Add global attention for specific tokens
    global_positions = [0, seq_len // 4, seq_len // 2, 3 * seq_len // 4]
    for pos in global_positions:
        if pos < seq_len:
            longformer_attention[pos, :] = 1  # Global token can attend to all
            longformer_attention[:, pos] = 1  # All tokens can attend to global
    
    im3 = axes[0, 2].imshow(longformer_attention.numpy(), cmap='Blues', aspect='auto')
    axes[0, 2].set_title('Longformer Attention\n(Sliding + Global)')
    axes[0, 2].set_xlabel('Key Positions')
    axes[0, 2].set_ylabel('Query Positions')
    # Mark global positions
    for pos in global_positions:
        if pos < seq_len:
            axes[0, 2].axhline(y=pos, color='red', linestyle='--', alpha=0.7)
            axes[0, 2].axvline(x=pos, color='red', linestyle='--', alpha=0.7)
    plt.colorbar(im3, ax=axes[0, 2])
    
    # Complexity comparison
    seq_lengths = [128, 256, 512, 1024, 2048, 4096]
    std_complexity = [n**2 for n in seq_lengths]
    sliding_complexity = [n * window_size for n in seq_lengths]
    longformer_complexity = [n * window_size + 4 * n for n in seq_lengths]  # 4 global tokens
    
    axes[1, 0].plot(seq_lengths, std_complexity, 'b-', label='Standard O(n²)', linewidth=2)
    axes[1, 0].plot(seq_lengths, sliding_complexity, 'g-', label=f'Sliding Window O(n×{window_size})', linewidth=2)
    axes[1, 0].plot(seq_lengths, longformer_complexity, 'r-', label='Longformer O(n×w + g×n)', linewidth=2)
    axes[1, 0].set_xlabel('Sequence Length')
    axes[1, 0].set_ylabel('Computational Operations')
    axes[1, 0].set_title('Computational Complexity Comparison')
    axes[1, 0].legend()
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Memory usage comparison
    memory_std = [n**2 * 4 / 1024**2 for n in seq_lengths]  # MB for attention matrix
    memory_sliding = [n * window_size * 4 / 1024**2 for n in seq_lengths]
    memory_longformer = [(n * window_size + 4 * n) * 4 / 1024**2 for n in seq_lengths]
    
    axes[1, 1].plot(seq_lengths, memory_std, 'b-', label='Standard', linewidth=2)
    axes[1, 1].plot(seq_lengths, memory_sliding, 'g-', label='Sliding Window', linewidth=2)
    axes[1, 1].plot(seq_lengths, memory_longformer, 'r-', label='Longformer', linewidth=2)
    axes[1, 1].set_xlabel('Sequence Length')
    axes[1, 1].set_ylabel('Memory Usage (MB)')
    axes[1, 1].set_title('Memory Usage Comparison')
    axes[1, 1].legend()
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Speedup visualization
    speedup_sliding = [std / sliding for std, sliding in zip(std_complexity, sliding_complexity)]
    speedup_longformer = [std / lf for std, lf in zip(std_complexity, longformer_complexity)]
    
    axes[1, 2].plot(seq_lengths, speedup_sliding, 'g-', label='Sliding Window vs Standard', linewidth=2)
    axes[1, 2].plot(seq_lengths, speedup_longformer, 'r-', label='Longformer vs Standard', linewidth=2)
    axes[1, 2].set_xlabel('Sequence Length')
    axes[1, 2].set_ylabel('Speedup Factor')
    axes[1, 2].set_title('Theoretical Speedup')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def analyze_attention_sparsity():
    """Analyze attention sparsity patterns"""
    print("\nAttention Sparsity Analysis")
    print("=" * 50)
    
    seq_len = 128
    window_sizes = [8, 16, 32, 64]
    
    fig, axes = plt.subplots(1, len(window_sizes), figsize=(20, 4))
    
    for i, window_size in enumerate(window_sizes):
        # Create sliding window pattern
        attention_matrix = torch.zeros(seq_len, seq_len)
        for j in range(seq_len):
            start = max(0, j - window_size // 2)
            end = min(seq_len, j + window_size // 2 + 1)
            attention_matrix[j, start:end] = 1
        
        # Add global tokens
        global_positions = [0, seq_len // 2]
        for pos in global_positions:
            attention_matrix[pos, :] = 1
            attention_matrix[:, pos] = 1
        
        # Calculate sparsity
        total_elements = seq_len * seq_len
        non_zero_elements = (attention_matrix > 0).sum().item()
        sparsity = 1 - (non_zero_elements / total_elements)
        
        im = axes[i].imshow(attention_matrix.numpy(), cmap='Blues', aspect='auto')
        axes[i].set_title(f'Window Size: {window_size}\nSparsity: {sparsity:.2%}')
        axes[i].set_xlabel('Key Positions')
        if i == 0:
            axes[i].set_ylabel('Query Positions')
        
        # Mark global positions
        for pos in global_positions:
            axes[i].axhline(y=pos, color='red', linestyle='--', alpha=0.7)
            axes[i].axvline(x=pos, color='red', linestyle='--', alpha=0.7)
        
        plt.colorbar(im, ax=axes[i])
    
    plt.tight_layout()
    plt.show()
    
    print("Key Observations:")
    print("• Smaller windows = Higher sparsity = More efficient")
    print("• Global tokens maintain long-range connections")
    print("• Trade-off between efficiency and expressiveness")

print("✓ Dataset and utilities created successfully")

In [ ]:
# Cell 4: Run Visualizations
print("Running Longformer concept visualizations...")

# Execute the visualization functions
visualize_attention_patterns()
analyze_attention_sparsity()

print("\n" + "="*60)
print("✅ All visualizations completed successfully!")
print("This demonstrates Longformer's key innovations:")
print("• Sliding window attention for local dependencies")
print("• Global attention for long-range connections")
print("• Significant reduction in computational complexity")
print("• Sparse attention patterns for efficiency")

In [ ]:
# Cell 5: Training Setup and Configuration
print("Setting up Longformer training...")

# Model configuration
config = {
    'vocab_size': 1000,
    'd_model': 256,
    'n_heads': 8,
    'n_layers': 4,
    'd_ff': 1024,
    'window_size': 32,
    'max_seq_len': 512,
    'num_classes': 2,
    'dropout': 0.1
}

# Training configuration
train_config = {
    'batch_size': 8,       # Smaller batch for longer sequences
    'seq_len': 256,        # Manageable sequence length
    'num_samples': 2000,   # Reasonable dataset size
    'num_epochs': 5,       # Fewer epochs for demonstration
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    'warmup_steps': 100,
    'eval_interval': 1,
    'log_interval': 10
}

# Create dataset
print("Creating long sequence dataset...")
dataset = LongSequenceDataset(
    num_samples=train_config['num_samples'],
    seq_len=train_config['seq_len'],
    vocab_size=config['vocab_size'],
    num_classes=config['num_classes']
)

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=train_config['batch_size'], 
    shuffle=True,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=train_config['batch_size'], 
    shuffle=False,
    num_workers=0
)

# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model
model = EfficientLongformer(
    vocab_size=config['vocab_size'],
    d_model=config['d_model'],
    n_heads=config['n_heads'],
    n_layers=config['n_layers'],
    d_ff=config['d_ff'],
    window_size=config['window_size'],
    max_seq_len=config['max_seq_len'],
    num_classes=config['num_classes'],
    dropout=config['dropout']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024**2:.2f} MB")

# Initialize optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=train_config['learning_rate'],
    weight_decay=train_config['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=train_config['num_epochs']
)

# Loss function
criterion = nn.CrossEntropyLoss()

print("✓ Training setup completed")

In [ ]:
# Cell 6: Training Loop and Execution
print("Starting Longformer training...")

def train_epoch(model, train_loader, optimizer, criterion, device, config):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (input_ids, labels) in enumerate(train_loader):
        try:
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            # Create attention masks
            attention_mask, global_attention_mask = create_attention_masks(
                input_ids, global_token_id=config['vocab_size'] - 1
            )
            attention_mask = attention_mask.to(device)
            global_attention_mask = global_attention_mask.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(input_ids, attention_mask, global_attention_mask)
            logits = outputs['logits']
            
            # Compute loss
            loss = criterion(logits, labels)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Statistics
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if batch_idx % config['log_interval'] == 0:
                print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
                
        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            continue
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

def evaluate_model(model, val_loader, criterion, device, config):
    """Evaluate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for input_ids, labels in val_loader:
            try:
                input_ids = input_ids.to(device)
                labels = labels.to(device)
                
                # Create attention masks
                attention_mask, global_attention_mask = create_attention_masks(
                    input_ids, global_token_id=config['vocab_size'] - 1
                )
                attention_mask = attention_mask.to(device)
                global_attention_mask = global_attention_mask.to(device)
                
                # Forward pass
                outputs = model(input_ids, attention_mask, global_attention_mask)
                logits = outputs['logits']
                
                # Compute loss
                loss = criterion(logits, labels)
                
                # Statistics
                total_loss += loss.item()
                _, predicted = torch.max(logits.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
            except Exception as e:
                continue
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

# Training loop
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

print(f"Training for {train_config['num_epochs']} epochs...")

for epoch in range(train_config['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{train_config['num_epochs']}")
    print("-" * 50)
    
    # Training
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, config)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # Validation
    val_loss, val_acc = evaluate_model(model, val_loader, criterion, device, config)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Update learning rate
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")

print("\n✓ Training completed!")

# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
epochs = range(1, len(train_losses) + 1)
axes[0].plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
axes[0].plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Progress - Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
axes[1].plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Progress - Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final train accuracy: {train_accuracies[-1]:.2f}%")
print(f"Final validation loss: {val_losses[-1]:.4f}")
print(f"Final validation accuracy: {val_accuracies[-1]:.2f}%")

print("\n✅ Longformer training completed successfully!")

In [ ]:
# Cell 7: Model Analysis and Attention Visualization
print("Creating model analysis and attention visualization...")

def visualize_learned_attention(model, sample_input, device, config):
    """Visualize the learned attention patterns"""
    model.eval()
    
    with torch.no_grad():
        input_ids = sample_input.unsqueeze(0).to(device)
        attention_mask, global_attention_mask = create_attention_masks(
            input_ids, global_token_id=config['vocab_size'] - 1
        )
        attention_mask = attention_mask.to(device)
        global_attention_mask = global_attention_mask.to(device)
        
        # Forward pass to get attention weights
        outputs = model(input_ids, attention_mask, global_attention_mask)
        sliding_weights = outputs['sliding_attention_weights']
        global_weights = outputs['global_attention_weights']
        
        seq_len = input_ids.size(1)
        
        # Visualize sliding window attention from the last layer
        if sliding_weights and sliding_weights[-1] is not None:
            last_sliding_attn = sliding_weights[-1][0, 0]  # First head of last layer
            
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            
            # Sliding attention heatmap (limited window)
            window_size = last_sliding_attn.size(-1)
            attn_matrix = torch.zeros(seq_len, seq_len)
            
            for i in range(seq_len):
                start_pos = max(0, i - window_size // 2)
                end_pos = min(seq_len, start_pos + window_size)
                window_attn = last_sliding_attn[i, :end_pos-start_pos]
                attn_matrix[i, start_pos:end_pos] = window_attn
            
            im1 = axes[0, 0].imshow(attn_matrix.cpu().numpy(), cmap='Blues', aspect='auto')
            axes[0, 0].set_title('Learned Sliding Window Attention\n(Last Layer, Head 1)')
            axes[0, 0].set_xlabel('Key Positions')
            axes[0, 0].set_ylabel('Query Positions')
            plt.colorbar(im1, ax=axes[0, 0])
            
            # Mark global token positions
            global_positions = torch.nonzero(global_attention_mask[0], as_tuple=False).squeeze(-1)
            for pos in global_positions:
                axes[0, 0].axhline(y=pos.item(), color='red', linestyle='--', alpha=0.7)
                axes[0, 0].axvline(x=pos.item(), color='red', linestyle='--', alpha=0.7)
        
        # Attention entropy analysis
        if sliding_weights:
            entropies = []
            for layer_attn in sliding_weights:
                if layer_attn is not None:
                    layer_entropy = -(layer_attn * torch.log(layer_attn + 1e-8)).sum(dim=-1).mean()
                    entropies.append(layer_entropy.cpu().item())
            
            axes[0, 1].plot(range(1, len(entropies) + 1), entropies, 'b-o', linewidth=2)
            axes[0, 1].set_xlabel('Layer')
            axes[0, 1].set_ylabel('Attention Entropy')
            axes[0, 1].set_title('Attention Entropy Across Layers')
            axes[0, 1].grid(True, alpha=0.3)
        
        # Token importance analysis
        input_tokens = input_ids[0].cpu().numpy()
        token_importance = torch.zeros(seq_len)
        
        # Calculate importance based on attention received
        if sliding_weights and sliding_weights[-1] is not None:
            last_attn = sliding_weights[-1][0]  # All heads of last layer
            # Sum attention weights received by each token
            for i in range(seq_len):
                importance = 0
                for j in range(seq_len):
                    for head in range(last_attn.size(0)):
                        window_center = min(last_attn.size(-1) // 2, 
                                          abs(i - j))
                        if window_center < last_attn.size(-1):
                            importance += last_attn[head, j, window_center].item()
                token_importance[i] = importance
        
        axes[1, 0].bar(range(seq_len), token_importance.numpy())
        axes[1, 0].set_xlabel('Token Position')
        axes[1, 0].set_ylabel('Attention Importance')
        axes[1, 0].set_title('Token Importance Based on Attention')
        
        # Mark special tokens
        cls_pos = 0
        global_token_id = config['vocab_size'] - 1
        special_positions = [cls_pos] + [i for i, token in enumerate(input_tokens) 
                                       if token == global_token_id]
        
        for pos in special_positions:
            axes[1, 0].axvline(x=pos, color='red', linestyle='--', alpha=0.7)
        
        # Sequence analysis
        axes[1, 1].text(0.1, 0.9, f'Sequence Length: {seq_len}', transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.1, 0.8, f'Window Size: {config["window_size"]}', transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.1, 0.7, f'Global Tokens: {len(special_positions)}', transform=axes[1, 1].transAxes)
        axes[1, 1].text(0.1, 0.6, f'CLS Token: Position {cls_pos}', transform=axes[1, 1].transAxes)
        
        # Show token sequence snippet
        token_snippet = input_tokens[:20]
        axes[1, 1].text(0.1, 0.4, f'First 20 tokens:\n{token_snippet}', 
                       transform=axes[1, 1].transAxes, fontsize=8)
        
        axes[1, 1].set_title('Sequence Analysis')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()

def analyze_computational_efficiency():
    """Analyze computational efficiency of Longformer vs standard attention"""
    print("\nComputational Efficiency Analysis")
    print("=" * 50)
    
    seq_lengths = [128, 256, 512, 1024, 2048, 4096]
    window_size = config['window_size']
    n_global = 4  # Typical number of global tokens
    
    # Time complexity analysis
    standard_ops = [n**2 for n in seq_lengths]
    longformer_ops = [n * window_size + n_global * n for n in seq_lengths]
    
    # Memory complexity analysis
    standard_memory = [n**2 * 4 / 1024**2 for n in seq_lengths]  # MB
    longformer_memory = [(n * window_size + n_global * n) * 4 / 1024**2 for n in seq_lengths]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Operations comparison
    axes[0].plot(seq_lengths, standard_ops, 'b-', label='Standard O(n²)', linewidth=2)
    axes[0].plot(seq_lengths, longformer_ops, 'r-', label=f'Longformer O(n×{window_size} + {n_global}×n)', linewidth=2)
    axes[0].set_xlabel('Sequence Length')
    axes[0].set_ylabel('Operations')
    axes[0].set_title('Computational Complexity')
    axes[0].legend()
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)
    
    # Memory comparison
    axes[1].plot(seq_lengths, standard_memory, 'b-', label='Standard', linewidth=2)
    axes[1].plot(seq_lengths, longformer_memory, 'r-', label='Longformer', linewidth=2)
    axes[1].set_xlabel('Sequence Length')
    axes[1].set_ylabel('Memory Usage (MB)')
    axes[1].set_title('Memory Requirements')
    axes[1].legend()
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)
    
    # Speedup factor
    speedup = [std / lf for std, lf in zip(standard_ops, longformer_ops)]
    axes[2].plot(seq_lengths, speedup, 'g-', linewidth=2)
    axes[2].set_xlabel('Sequence Length')
    axes[2].set_ylabel('Speedup Factor')
    axes[2].set_title('Longformer Speedup vs Standard Attention')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print efficiency gains
    print("Efficiency Gains at Different Sequence Lengths:")
    for i, seq_len in enumerate(seq_lengths):
        if i < len(speedup):
            print(f"  {seq_len:4d} tokens: {speedup[i]:.1f}x speedup, "
                  f"{standard_memory[i]/longformer_memory[i]:.1f}x memory reduction")

def demonstrate_global_vs_local_attention():
    """Demonstrate the difference between global and local attention"""
    print("\nGlobal vs Local Attention Demonstration")
    print("=" * 50)
    
    # Create a sample sequence
    seq_len = 64
    sample_seq = torch.randint(2, config['vocab_size'] - 1, (seq_len,))
    sample_seq[0] = 1  # CLS token
    sample_seq[seq_len//4] = config['vocab_size'] - 1  # Global token
    sample_seq[seq_len//2] = config['vocab_size'] - 1  # Global token
    sample_seq[3*seq_len//4] = config['vocab_size'] - 1  # Global token
    
    print(f"Sample sequence created with {seq_len} tokens")
    print(f"Global tokens at positions: [0, {seq_len//4}, {seq_len//2}, {3*seq_len//4}]")
    
    # Visualize learned attention for this sample
    visualize_learned_attention(model, sample_seq, device, config)

# Run all analyses
print("Running comprehensive model analysis...")

# Computational efficiency analysis
analyze_computational_efficiency()

# Global vs local attention demonstration
demonstrate_global_vs_local_attention()

print("\n✅ Model analysis completed!")
print("\nKey Insights:")
print("• Longformer significantly reduces computational complexity")
print("• Sliding window captures local dependencies efficiently")
print("• Global tokens enable long-range information flow")
print("• Sparse attention patterns maintain model expressiveness")
print("• Memory usage scales linearly instead of quadratically")

# Clarification on the Longformer Implementation

The provided implementation is not a full, authentic Longformer as described in the original paper by Beltagy et al. Instead, it's a simplified approximation that demonstrates one key aspect of the Longformer: the sliding window attention mechanism.

## Key Differences from True Longformer

1. **Attention Mechanism**: The true Longformer combines sliding window attention with global attention on specific tokens. This implementation only includes the sliding window part.

2. **Efficiency**: The actual Longformer uses specialized CUDA kernels for efficient implementation. This code is a naive Python implementation that doesn't capture the full computational benefits of the Longformer.

3. **Scale**: Longformers are designed to handle much longer sequences (thousands or tens of thousands of tokens) than what this simple implementation can efficiently manage.

4. **Global Attention**: The true Longformer allows for global attention on certain tokens, which this implementation doesn't include.

5. **Dilated Sliding Window**: The actual Longformer can use a dilated sliding window for even longer-range dependencies, which this implementation doesn't have.

## Accurate Description

What's provided is more accurately described as a "sliding window attention mechanism inspired by Longformer". It demonstrates the concept of limiting attention to a local window, but it doesn't capture the full complexity and efficiency of the true Longformer.

For an authentic Longformer implementation, it's recommended to use the official implementation provided by the authors or a well-maintained library like Hugging Face Transformers.